# 03 · DDIM inversion & prompt editing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHITCRAFTSYT/difflab/blob/main/notebooks/03_ddim_inversion.ipynb)

Invert a real image to its latent with Stable Diffusion, then edit it by changing the prompt. **Needs a GPU runtime** to download/run SD.

In [ ]:
# === difflab bootstrap — makes `import difflab` and the configs work anywhere ===
# Works on Colab/Kaggle (no checkout) and on a local clone. Three install paths,
# tried in order: already-installed -> GitHub clone -> uploaded difflab.zip.
import glob, os, pathlib, subprocess, sys

def _have_difflab():
    try:
        import difflab  # noqa: F401
        return True
    except ModuleNotFoundError:
        return False

ROOT = None
if not _have_difflab():
    # (1) If you pushed difflab to GitHub, set DIFFLAB_REPO and it clones+installs:
    REPO_URL = os.environ.get('DIFFLAB_REPO', '')  # e.g. 'https://github.com/you/difflab.git'
    if REPO_URL:
        subprocess.run(['git', 'clone', '-q', REPO_URL, '/content/difflab'], check=False)
        ROOT = '/content/difflab'
    else:
        # (2) Else upload difflab.zip via the Files panel (left sidebar), then run this.
        hits = glob.glob('/content/difflab.zip') or glob.glob('difflab.zip')
        if not hits:
            from google.colab import files  # type: ignore
            print('Select difflab.zip to upload...')
            files.upload()
            hits = glob.glob('difflab.zip') or glob.glob('/content/difflab.zip')
        subprocess.run(['unzip', '-q', '-o', hits[0], '-d', '/content/difflab_pkg'], check=False)
        found = glob.glob('/content/difflab_pkg/**/pyproject.toml', recursive=True)
        ROOT = str(pathlib.Path(found[0]).parent) if found else '/content/difflab_pkg'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', ROOT], check=False)

# Resolve the repo root so we can find configs/ whether installed remotely or locally.
if ROOT is None:
    here = pathlib.Path.cwd()
    ROOT = str(here.parent if here.name == 'notebooks' else here)
CONFIGS = str(pathlib.Path(ROOT) / 'configs')

import difflab, torch
print('difflab', difflab.__version__, '| configs:', CONFIGS)
print('CUDA available:', torch.cuda.is_available())

## The inversion math (CPU, no download)
First verify the deterministic invert→sample round-trip on a tiny model — the same code path used for Stable Diffusion.

In [ ]:
import torch
from difflab.config import ModelConfig, SchedulerConfig
from difflab.models import build_unet, build_scheduler
from difflab.inversion import ddim_invert, ddim_sample_latents

unet = build_unet(ModelConfig(sample_size=8, in_channels=1, out_channels=1,
    layers_per_block=1, norm_num_groups=8, block_out_channels=(8,16),
    down_block_types=('DownBlock2D','DownBlock2D'),
    up_block_types=('UpBlock2D','UpBlock2D')))
sched = build_scheduler(SchedulerConfig(num_train_timesteps=50), 'ddim',
    clip_sample=False, set_alpha_to_one=False)
x0 = torch.randn(1,1,8,8)
const = torch.randn(1,1,8,8)
eps = lambda l,t: const.expand_as(l)
noise = ddim_invert(eps, sched, x0, 50)
recon = ddim_sample_latents(eps, sched, noise, 50)
print('round-trip rel error:', (recon-x0).norm().item()/x0.norm().item())

## Real-image editing with Stable Diffusion (GPU)
Uncomment to invert a real image and swap its content via the target prompt.

In [ ]:
# from PIL import Image
# from difflab.inversion import DDIMInverter
# inv = DDIMInverter.from_pretrained('runwayml/stable-diffusion-v1-5', device='cuda',
#                                    num_inference_steps=50)
# image = Image.open('cat.png').convert('RGB').resize((512,512))
# latents = inv.invert(image, prompt='a photo of a cat')
# edited  = inv.sample(latents, prompt='a photo of a dog')
# edited[0]